In [ ]:
import nest_asyncio
nest_asyncio.apply()

from wakis import SolverFIT3D
#from wakis import GridFIT3D 
from wakis import WakeSolver

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

#v.set_jupyter_backend('html') 
pv.set_jupyter_backend('trame') 

from benchmark import benchmark

### Check that the environment wakis-env succesfully installe the space mkl multithreading packages

In [ ]:
import os
print(f"Threads available: {os.cpu_count()}")

In [ ]:
pv.Report()

### 1) Select_interior_points from PyVista 0.47, instead of select_enclosed_points form PyVista 0.45/46 (In Wakis!)

https://docs.pyvista.org/api/core/_autosummary/pyvista.datasetfilters.select_interior_points

### 2) Compute implicit distance (PyVista 0.24.0 , to define the surface)

https://docs.pyvista.org/api/core/_autosummary/pyvista.datasetfilters.compute_implicit_distance

### 3) PyVista 0.46.0, Voxelize

https://docs.pyvista.org/api/core/_autosummary/pyvista.datasetfilters.voxelize_rectilinear

# Cavity (benchmark known structure)

#### How it's done in Wakis:

In [ ]:

stl_cavity = 'examples/data/001_vacuum_cavity.stl' 
stl_shell = 'examples/data/001_lossymetal_shell.stl' 
surf_shell=pv.read(stl_shell) 

# Set up geometry & materials dictionaries
stl_solids = {'cavity': stl_cavity, 'shell': stl_shell}
stl_materials = {'cavity': 'vacuum', 'shell': [30, 1.0, 30]}

# Domain bounds
surf = pv.read(stl_shell) + pv.read(stl_cavity)
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)

Nx, Ny, Nz = 100, 100, 200  
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
dx = np.diff(x)
dy = np.diff(y)
dz = np.diff(z)
stl_tol=1e-3
stl_tolerance = np.min([np.min(dx), np.min(dy), np.min(dz)]) *stl_tol


grid = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, 
                stl_solids=stl_solids, 
                stl_materials=stl_materials,
                stl_scale=1.0
                #use_mesh_refinement=True,
)

grid.plot_stl_mask(stl_solid='shell', stl_opacity=0.2, ymax=0)




In [ ]:
grid._mark_cells_in_stl()

#spacing = [dx[0],dy[0],dz[0]]
#print(spacing)

xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
print([(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200])  #winner (0 volume missing to be gridded)
spacing=[(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]

benchmark(grid.grid,spacing,surf_shell)


In [ ]:
print('extracted area of the full grid:',grid.grid.extract_surface().area)
is_inside=grid.grid['shell'].view(np.bool_)
print('extracted surface of the meshed device i.e. only the mask:',grid.grid.extract_cells(is_inside).extract_surface().area)

#The overall grid surface 
pl=pv.Plotter()
pl.add_mesh(grid.grid.extract_surface())
pl.show()

#extracted surface of the meshed device
pl=pv.Plotter()
pl.add_mesh(grid.grid.extract_cells(is_inside).extract_surface())
pl.show()

#extracted surface of the stl file
pl=pv.Plotter()
pl.add_mesh(surf_shell)
pl.show()

In [ ]:
inside2= grid.grid.extract_cells(is_inside)
inside2.clear_cell_data()
#inside2 = inside2.triangulate()
inside2.reconstruct_surface().plot()

In [ ]:
print(np.sum(grid.grid['shell'].view(np.bool_)))

### Manually as in Wakis

Differences:
1. My simplified version below uses pv.Rectlineargrid and not pv.Structuredgrid
2. -||- does not have mesh refinement as wakis with:

        elf._compute_snap_points(snap_solids=snap_solids, snap_tol=snap_tol)
                self._refine_xyz_axis(method=refinement_method, tol=refinement_tol)

3. -||- does not allow use of non-uniform grid, and calculates dx, dy and dz as dz below:


        #TODO: allow non uniform dx, dy, dz
                self.dx = np.min(np.diff(self.x))
                self.dy = np.min(np.diff(self.y))
                #self.dz = np.min(np.diff(self.z))
                self.dz = (self.zmax - self.zmin)/self.Nz

In [ ]:
stl_cavity = 'examples/data/001_vacuum_cavity.stl' 
stl_shell = 'examples/data/001_lossymetal_shell.stl'
surf = pv.read(stl_cavity) + pv.read(stl_shell)
surf_shell = pv.read(stl_shell) 

Nx, Ny, Nz = 100, 100, 200  
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
grid = pv.RectilinearGrid(x, y, z)

spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]
dx = np.diff(x)
dy = np.diff(y)
dz = np.diff(z)
stl_tol=1e-3
stl_tolerance = np.min(spacing) *stl_tol


select = grid.select_enclosed_points(surf_shell, tolerance=stl_tolerance, check_surface=False)

# GridFIT3D expects the mask to be stored as CELL DATA in the grid
# point_data_to_cell_data() converts the 0/1 point values to a cell average
key = 'shell' # This is your 'stl_solid' name
grid[key] = select.point_data_to_cell_data()['SelectedPoints'] > stl_tolerance


pl = pv.Plotter()
pl.add_mesh(
    grid, 
    scalars=key, 
    cmap='viridis', 
    opacity=0.5, 
    show_edges=True
)
pl.add_mesh(surf, color='white', opacity=0.1, style='wireframe')
pl.show()



In [ ]:
benchmark(grid,spacing,surf_shell)   

In [ ]:
is_inside=grid[key].view(np.bool_)

In [ ]:
inside2= grid.extract_cells(is_inside)

pl=pv.Plotter()
pl.add_mesh(inside2)
pl.show()



In [ ]:
shell_surf = inside2.extract_surface(algorithm='dataset_surface')
reconstructed = shell_surf.reconstruct_surface()
accurate_area = reconstructed.area
print(accurate_area)

In [ ]:
inside2.clear_cell_data()

pl=pv.Plotter()
pl.add_mesh(inside2)
pl.show()

In [ ]:
inside3=inside2

In [ ]:
inside2=inside2.triangulate()


In [ ]:
inside2.plot()

In [ ]:
inside2.area

In [ ]:
#TO TRY

# Instead of extract_cells, use contour
# This assumes your 'shell' data is 0 and 1
'''iso_surface = shell_surf.contour(isosurfaces=[0.5], scalars=key)
pl=pv.Plotter()
pl.add_mesh(iso_surface)
pl.show()
better_area = iso_surface.area'''

In [ ]:
#inside3=inside3.reconstruct_surface()     It needs point_data
#inside3.area
#inside3.plot()

### TEST OF THE SELECT_ENCLOSED_POINT MANUAL/primitive version of wakis: does it give consistent results?

In [ ]:
stl_cavity = 'examples/data/001_vacuum_cavity.stl' 
stl_shell = 'examples/data/001_lossymetal_shell.stl'
surf=pv.read(stl_cavity)+pv.read(stl_shell)
surf_shell= pv.read(stl_shell) 

Nx, Ny, Nz = 100, 100, 200  
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
grid = pv.RectilinearGrid(x, y, z)

spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]
dx = np.diff(x)
dy = np.diff(y)
dz = np.diff(z)
stl_tol=1e-3
stl_tolerance = np.min(spacing) *stl_tol


key='mask'  #when eveerything works, this should be changes to a Wakis (GridFIT3D) compatible string, as this is currently 'shell' in wakis.

select = grid.select_enclosed_points(surf_shell, tolerance=stl_tolerance, check_surface=False)
grid_cells=select.point_data_to_cell_data()
grid_cells.cell_data[key]= grid_cells.cell_data['SelectedPoints'] > stl_tolerance
#grid_cells.cell_data[key]= grid_cells.cell_data['SelectedPoints'] > 0.5

# threshold only works on a mesh object, and the point_data_to_cell_data returns a flat numpy array we must store the cell mask as a mesh/grid (3D)
inside = grid_cells.threshold(0.5, scalars=key)


pl = pv.Plotter()
pl.add_mesh(surf, style='wireframe', opacity=0.2)
pl.add_mesh(inside, color='red', show_edges=True)
pl.show()



In [ ]:
benchmark(grid_cells,spacing,surf_shell)   

# The other methods 

# 1

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
key='mask'
select = grid.select_interior_points(surf_shell,check_surface=False)
grid[key]= select.point_data['selected_points'] > stl_tolerance


#is_inside = select.point_data['selected_points'].view(np.bool_)

#grid.cell_data[key]=select.point_data['selected_points'] > stl_tolerance

inside = grid.threshold(0.5, scalars=key)


In [ ]:
grid_cells=select.point_data_to_cell_data()
grid_cells.cell_data[key]= grid_cells.cell_data['selected_points'] > stl_tolerance

benchmark(grid_cells,spacing,surf_shell)   

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
select = grid.select_interior_points(surf_shell,check_surface=False)
grid['mask'] = select.point_data_to_cell_data()['selected_points'] > stl_tolerance
benchmark(grid,spacing,surf_shell)   



In [ ]:
grid = pv.RectilinearGrid(x, y, z)
select = grid.select_interior_points(surf_shell,check_surface=False)
grid['mask'] = select.point_data_to_cell_data()['selected_points'] > 0.5
benchmark(grid,spacing,surf_shell)   

# 2

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from wakis import SolverFIT3D
from wakis import GridFIT3D 
from wakis import WakeSolver

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

pv.set_jupyter_backend('html') 
#pv.set_jupyter_backend('trame') 

from benchmark import benchmark

In [ ]:

stl_cavity = 'examples/data/001_vacuum_cavity.stl' 
stl_shell = 'examples/data/001_lossymetal_shell.stl' 
surf_shell=pv.read(stl_shell) 

# Set up geometry & materials dictionaries
stl_solids = {'cavity': stl_cavity, 'shell': stl_shell}
stl_materials = {'cavity': 'vacuum', 'shell': [30, 1.0, 30]}

# Domain bounds
surf = pv.read(stl_shell) + pv.read(stl_cavity)
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)

Nx, Ny, Nz = 100, 100, 200  
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)
dx = np.diff(x)
dy = np.diff(y)
dz = np.diff(z)
stl_tol=1e-3
stl_tolerance = np.min([np.min(dx), np.min(dy), np.min(dz)]) *stl_tol

spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
grid1=grid.copy()
grid2=grid.copy()

dist=grid2.compute_implicit_distance(surf_shell, inplace=True)
grid1.compute_implicit_distance(surf_shell, inplace=True)

#grid1['mask']= (dist["implicit_distance"] < 0).
grid1['mask']=dist.point_data_to_cell_data()['implicit_distance'] < 0
print(grid1)
print(grid1['mask'])

#grid2['mask']= (dist["implicit_distance"] < stl_tolerance).astype(bool)
grid2['mask']= dist.point_data_to_cell_data()['implicit_distance'] < stl_tolerance
print(grid2)
print(grid2['mask'])

print('benchmarking grid1')
benchmark(grid1, spacing, surf_shell)   
print('benchmarking grid2')
benchmark(grid2, spacing, surf_shell)   

In [ ]:
grid = pv.RectilinearGrid(x, y, z)

grid.compute_implicit_distance(surf_shell, inplace=True)
dist = grid["implicit_distance"] 

'''
dx, dy, dz = np.diff(grid.x).mean(), np.diff(grid.y).mean(), np.diff(grid.z).mean()
epsilon = np.mean([dx, dy, dz]) * 0.5
material_weight = np.clip(0.5-(dist / (2*epsilon)),0,1)
grid["material_density"] = material_weight
'''

# Create binary flag: 1 for inside, 0 for outside
grid["mask"] = (grid["implicit_distance"] <= stl_tolerance).astype(float)
# Extract only the '1's
inside_voxels = grid.threshold(0.5, scalars="mask")

pl = pv.Plotter()
pl.add_mesh(surf, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')
#pl.add_mesh(dist, color='red', show_edges=True, label='Grid implicit distances')
pl.add_legend()
pl.show()


In [ ]:
#voxelized_grid = grid.point_data_to_cell_data()
#benchmark(voxelized_grid,spacing,surf_shell)   

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
grid.compute_implicit_distance(surf_shell, inplace=True)
grid_cells = grid.point_data_to_cell_data()
key="mask"
grid_cells[key] = (grid_cells["implicit_distance"] <= stl_tolerance).astype(float)

cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
grid_cells.plot(scalars='mask', show_edges=False, cpos=cpos)

benchmark(grid_cells, spacing, surf_shell)

# 3

In [ ]:
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]

voxelized_grid = surf_shell.voxelize_rectilinear(spacing=spacing)
print(voxelized_grid)
print(voxelized_grid['mask'])
print(voxelized_grid.cell_data['mask'])
cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
voxelized_grid.plot(scalars='mask', show_edges=False, cpos=cpos)
array_name = 'mask'
is_inside = voxelized_grid.cell_data[array_name].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf, color='cyan', style='wireframe', opacity=0.3,label="Origional STL")
inside_voxels = voxelized_grid.extract_cells(voxelized_grid.cell_data[array_name].view(np.bool_))
pl.add_mesh(inside_voxels, color='red', label="Voxelized Interior")
pl.add_legend()
pl.show()

In [ ]:
benchmark(voxelized_grid,spacing,surf_shell)

In [ ]:
voxelized_grid

# Benchmarking 

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from wakis import SolverFIT3D
#from wakis import GridFIT3D 
from wakis import WakeSolver

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

pv.set_jupyter_backend('html') 
#pv.set_jupyter_backend('trame') 

from benchmark import benchmark

In [ ]:
stl_cavity = 'examples/data/001_vacuum_cavity.stl' 
stl_shell = 'examples/data/001_lossymetal_shell.stl' 
surf_shell=pv.read(stl_shell) 
stl_solids = {'cavity': stl_cavity, 'shell': stl_shell}
stl_materials = {'cavity': 'vacuum', 'shell': [30, 1.0, 30]}

surf = pv.read(stl_shell) + pv.read(stl_cavity)
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)


stl_tol=1e-3

In [ ]:
methods = ['wakis','enclosed_points', 'interior_points','implicit_distance', 'voxelize_rectlinear']
results = {}
#resolutions = ['25x25x50', '50x50x100', '75x75x150','100x100x200', '125x125x250', '150x150x300', '175x175x350', '200x200x400'] 
resolutions = ['25x25x50', '50x50x100','100x100x200', '125x125x250', '150x150x300', '175x175x350'] 
results = {m: {} for m in methods}

for r in resolutions:
    Nx, Ny, Nz = int(r.split('x')[0]), int(r.split('x')[1]), int(r.split('x')[2]) 
    spacing=[(xmax - xmin) / Nx, (ymax - ymin) / Ny, (zmax - zmin) / Nz]
    x,y,z = np.linspace(xmin, xmax, Nx), np.linspace(ymin, ymax, Ny), np.linspace(zmin, zmax, Nz)
    stl_tolerance = np.min([np.min(np.diff(x)), np.min(np.diff(y)), np.min(np.diff(z))]) *stl_tol

    grid = pv.RectilinearGrid(x, y, z)

    print(f"----- Resolution: {r} -----")
    for m in methods:
        print(f"----- Benchmarking Method: {m} -----")
        
        test_grid = grid.copy()  
        
        if m == 'wakis':
            gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials,stl_scale=1.0)
            gfit._mark_cells_in_stl()
            test_grid = gfit.grid
            pass 
        
        elif m == 'enclosed_points': #test to see if same results as Wakis
            select = test_grid.select_enclosed_points(surf_shell, tolerance=stl_tolerance,check_surface=False)
            #test_grid['mask'] = select.point_data_to_cell_data()['SelectedPoints'] > 0.5
            test_grid['mask'] = select.point_data_to_cell_data()['SelectedPoints'] > stl_tolerance #To Do

        elif m == 'interior_points':  
            select = test_grid.select_interior_points(surf_shell, method='cell_locator',check_surface=False,locator_tolerance=stl_tolerance)
            #select = test_grid.select_interior_points(surf_shell, method='signed_distance',check_surface=False)
            #test_grid['mask'] = select.point_data_to_cell_data()['selected_points'] > 0.5 
            test_grid['mask'] = select.point_data_to_cell_data()['selected_points'] > stl_tolerance #TO DO

        elif m == 'implicit_distance':   #To DO: choose which threshold for the mask
            # Computes distance to surface; negative is inside
            dist = test_grid.compute_implicit_distance(surf_shell)
            #test_grid['mask'] = dist['implicit_distance'] < 0
            test_grid['mask'] = dist.point_data_to_cell_data()['implicit_distance'] < 0
            #test_grid['mask'] = dist.point_data_to_cell_data()['implicit_distance'] < stl_tolerance

        elif m == 'voxelize_rectlinear':
            # PyVista built-in voxelization
            # Note: voxelize returns a new grid, which will be used directly
            vox = surf_shell.voxelize_rectilinear(spacing=spacing)
            test_grid = vox
            #test_grid['mask'] = np.ones(test_grid.n_cells, dtype=bool)

        vol, area, errvol, errarea = benchmark(test_grid, spacing, surf_shell)
        
        results[m][r] = {'vol': vol, 'vol_err': errvol, 'area': area, 'area_err': errarea}


import pickle
filename = 'clara_pvMesh_benchmark_results_mistake.pkl'
with open(filename, 'wb') as f:
    pickle.dump(results, f)

print(f"Successfully saved results to {filename}")

# Benchmarking with methods implemented in Wakis :D

In [ ]:
import nest_asyncio
nest_asyncio.apply()

from wakis import WakeSolver
from wakis import SolverFIT3D
#from wakis import GridFIT3D 
#from  clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D
#from wakis import clara_gridFIT3D_markCellsinSTL_WIP 
from wakis.clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

pv.set_jupyter_backend('html') 
#pv.set_jupyter_backend('trame') 

from benchmark import benchmark

import pickle

In [ ]:
stl_cavity = 'examples/data/001_vacuum_cavity.stl' 
stl_shell = 'examples/data/001_lossymetal_shell.stl' 
surf_shell=pv.read(stl_shell) 
surf_cavity=pv.read(stl_cavity) 
#stl_solids = {'cavity': stl_cavity, 'shell': stl_shell}
#stl_materials = {'cavity': 'vacuum', 'shell': [30, 1.0, 30]}

surf = pv.read(stl_shell) + pv.read(stl_cavity)
xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)

xmin, xmax, ymin, ymax, zmin, zmax = surf.bounds
stl_tol=1e-3

In [ ]:
'''
methods = ['enclosed_points', 'interior_points','implicit_distance', 'voxelize_rectilinear']
results = {}
#resolutions = ['25x25x50', '50x50x100', '75x75x150','100x100x200', '125x125x250', '150x150x300', '175x175x350', '200x200x400'] 
resolutions = ['25x25x50', '50x50x100','100x100x200', '125x125x250', '150x150x300', '175x175x350'] 
results = {m: {} for m in methods}

for r in resolutions:
    Nx, Ny, Nz = int(r.split('x')[0]), int(r.split('x')[1]), int(r.split('x')[2]) 
    spacing=[(xmax - xmin) / Nx, (ymax - ymin) / Ny, (zmax - zmin) / Nz]
    x,y,z = np.linspace(xmin, xmax, Nx), np.linspace(ymin, ymax, Ny), np.linspace(zmin, zmax, Nz)
    stl_tolerance = np.min([np.min(np.diff(x)), np.min(np.diff(y)), np.min(np.diff(z))]) *stl_tol

    grid = pv.RectilinearGrid(x, y, z)

    print(f"----- Resolution: {r} -----")
    for m in methods:
        print(f"----- Benchmarking Method: {m} -----")
        
        test_grid = grid.copy()  
        
        if m == 'enclosed_points':
            gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials,stl_scale=1.0)
            gfit._mark_cells_in_stl(method='enclosed_points')
            test_grid = gfit.grid
            pass 

        elif m == 'interior_points':  
            gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials,stl_scale=1.0)
            gfit._mark_cells_in_stl(method='interior_points')
            test_grid = gfit.grid

        elif m == 'implicit_distance':   
            gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials,stl_scale=1.0)
            gfit._mark_cells_in_stl(method='implicit_distance')
            test_grid = gfit.grid

        elif m == 'voxelize_rectilinear':
            gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials,stl_scale=1.0)
            gfit._mark_cells_in_stl(method='voxelize_rectlinear')
            test_grid = gfit.grid

        vol, area, errvol, errarea = benchmark(test_grid, spacing, surf_shell)
        
        results[m][r] = {'vol': vol, 'vol_err': errvol, 'area': area, 'area_err': errarea}


import pickle
filename = 'clara_pvMesh_benchmark_results_mistake.pkl'
with open(filename, 'wb') as f:
    pickle.dump(results, f)

print(f"Successfully saved results to {filename}")
'''

In [ ]:
stl_solids = { 'Fingers': stl_finger}
stl_materials = {'Fingers': [1e4, 1., 1e4]}

Nx, Ny, Nz = 100, 100, 200

gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,stl_method='implicit_distance')

testgrid = gfit.grid
surf = gfit.read_stl('Fingers')
#spacing = testgrid.spacing
spacing = [gfit.x[1] - gfit.x[0], gfit.y[1] - gfit.y[0], gfit.z[1] - gfit.z[0]]



vol, area, errvol, errarea = benchmark(testgrid, spacing, surf_finger)
print(f"Volume: {vol}, Area: {area}, Volume Error: {errvol}, Area Error: {errarea}")



In [ ]:
vox = surf.voxelize_rectilinear(spacing=spacing)
#vox = surf.voxelize_rectilinear(reference_volume=self.grid)   #the self.grid is a structured grid, but voxelize?rectlinear expects rectlinear grid. TO DO/TEST
testgrid['shell'] = vox.cell_data['mask'].astype(bool)


vox = surf.voxelize_rectilinear(reference_volume=gfit.grid)  
testgrid['shell'] = vox.cell_data['mask'].astype(bool)

vol, area, errvol, errarea = benchmark(testgrid, spacing, surf_shell)
print(f"Volume: {vol}, Area: {area}, Volume Error: {errvol}, Area Error: {errarea}")

Running in parallel

In [ ]:

from concurrent.futures import ProcessPoolExecutor
#benchmarking just works for one key at a time right now, so just for simplicity/testing:
stl_solids = { 'shell': stl_shell}
stl_materials = { 'shell': [30, 1.0, 30]}

# 1. Define the worker function
# This must be at the top level of the script so workers can find it
def run_benchmark_task(params):
    m, r, xmin, xmax, ymin, ymax, zmin, zmax, stl_solids, stl_materials, surf_shell = params
    
    Nx, Ny, Nz = map(int, r.split('x'))
    #spacing = [(xmax - xmin) / Nx, (ymax - ymin) / Ny, (zmax - zmin) / Nz]
    x, y, z = np.linspace(xmin, xmax, Nx), np.linspace(ymin, ymax, Ny), np.linspace(zmin, zmax, Nz)
    
    gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, 
                     stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,
                     stl_method=m)
    
    #git._mark_cells_in_stl() computes it double 
    test_grid = gfit.grid
    
    spacing = [gfit.x[1] - gfit.x[0], gfit.y[1] - gfit.y[0], gfit.z[1] - gfit.z[0]]
    vol, area, errvol, errarea = benchmark(test_grid, spacing, surf_shell)
    return m, r, {'vol': vol, 'vol_err': errvol, 'area': area, 'area_err': errarea}

if __name__ == '__main__':
    methods = ['enclosed_points', 'interior_points', 'interior_points_signed', 'implicit_distance', 'voxelize_rectilinear']
    #methods = ['enclosed_points', 'interior_points', 'implicit_distance']
    resolutions = ['25x25x50', '50x50x100', '100x100x200', '125x125x250', '150x150x300', '175x175x350']

    tasks = []
    for r in resolutions:
        for m in methods:
            # Pass all necessary variables to the worker
            tasks.append((m, r, xmin, xmax, ymin, ymax, zmin, zmax, 
                          stl_solids, stl_materials, surf_shell))

    results = {m: {} for m in methods}

    print(f"Starting benchmark on 12 cores for {len(tasks)} tasks...")
    with ProcessPoolExecutor(max_workers=12) as executor:
        for m, r, data in executor.map(run_benchmark_task, tasks):
            results[m][r] = data
            print(f"Finished: {m} at {r}")

    
    filename = 'clara_pvMesh_benchmark_results_8_test1.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

    print(f"Successfully saved results to {filename}")

# The finger

In [4]:
import nest_asyncio
nest_asyncio.apply()

from wakis import WakeSolver
from wakis import SolverFIT3D
#from wakis import GridFIT3D 
#from  clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D
#from wakis import clara_gridFIT3D_markCellsinSTL_WIP 
from wakis.clara_gridFIT3D_markCellsinSTL_WIP import GridFIT3D

import numpy as np
import pyvista as pv

from pyvista.trame.jupyter import launch_server
await launch_server().ready

#pv.set_jupyter_backend('html') 
pv.set_jupyter_backend('trame') 

from benchmark import benchmark

import pickle

In [5]:
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)


materials = {'Fingers': [1e4, 1., 1e4]}
stl_names = {'Fingers': stl_finger}
basename = 'Fingers'
stl_solids = {m: f'{basename}-{stl_names[m]}.stl' for m in materials}


xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)
stl_tol=1e-3

In [ ]:

from concurrent.futures import ProcessPoolExecutor
#benchmarking just works for one key at a time right now, so just for simplicity/testing:
stl_solids = { 'Fingers': stl_finger}
stl_materials = {'Fingers': [1e4, 1., 1e4]}


# 1. Define the worker function
# This must be at the top level of the script so workers can find it
def run_benchmark_task(params):
    m, r, xmin, xmax, ymin, ymax, zmin, zmax, stl_solids, stl_materials, surf_shell = params
    
    Nx, Ny, Nz = map(int, r.split('x'))
    #spacing = [(xmax - xmin) / Nx, (ymax - ymin) / Ny, (zmax - zmin) / Nz]
    x, y, z = np.linspace(xmin, xmax, Nx), np.linspace(ymin, ymax, Ny), np.linspace(zmin, zmax, Nz)
    
    gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, 
                     stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,
                     stl_method=m)
    
    #git._mark_cells_in_stl() computes it double 
    test_grid = gfit.grid
    
    spacing = [gfit.x[1] - gfit.x[0], gfit.y[1] - gfit.y[0], gfit.z[1] - gfit.z[0]]
    vol, area, errvol, errarea = benchmark(test_grid, spacing, surf_finger)
    return m, r, {'vol': vol, 'vol_err': errvol, 'area': area, 'area_err': errarea}

if __name__ == '__main__':
    methods = ['enclosed_points', 'interior_points', 'interior_points_signed', 'implicit_distance', 'implicit_distance_tol', 'voxelize_rectilinear']
    #methods = ['enclosed_points', 'interior_points', 'implicit_distance']
    resolutions = ['25x25x50', '50x50x100', '100x100x200', '125x125x250', '150x150x300', '175x175x350']

    tasks = []
    for r in resolutions:
        for m in methods:
            # Pass all necessary variables to the worker
            tasks.append((m, r, xmin, xmax, ymin, ymax, zmin, zmax, 
                          stl_solids, stl_materials, surf_finger))

    results = {m: {} for m in methods}

    print(f"Starting benchmark on 12 cores for {len(tasks)} tasks...")
    with ProcessPoolExecutor(max_workers=12) as executor:
        for m, r, data in executor.map(run_benchmark_task, tasks):
            results[m][r] = data
            print(f"Finished: {m} at {r}")

    
    filename = 'clara_pvMesh_bench_finger_res_mis.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(results, f)

    print(f"Successfully saved results to {filename}")

In [ ]:
'''import pickle
filename = 'clara_pvMesh_bench_finger_res_0.pkl'
with open(filename, 'wb') as f:
    pickle.dump(results, f)

print(f"Successfully saved results to {filename}")'''

In [ ]:
stl_solids = { 'Fingers': stl_finger}
stl_materials = {'Fingers': [1e4, 1., 1e4]}

Nx, Ny, Nz = 100, 100, 200

gfit_imp = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,
                     stl_method='implicit_distance')

testgrid = gfit_imp.grid
surf = gfit_imp.read_stl('Fingers')
#spacing = testgrid.spacing
spacing = [gfit_imp.x[1] - gfit_imp.x[0], gfit_imp.y[1] - gfit_imp.y[0], gfit_imp.z[1] - gfit_imp.z[0]]

vol, area, errvol, errarea = benchmark(testgrid, spacing, surf_finger)
print(f"Volume: {vol}, Area: {area}, Volume Error: {errvol}, Area Error: {errarea}")



In [ ]:
stl_solids = { 'Fingers': stl_finger}
stl_materials = {'Fingers': [1e4, 1., 1e4]}
Nx, Ny, Nz = 100, 100, 200

gfit = GridFIT3D(xmin, xmax, ymin, ymax, zmin, zmax, Nx, Ny, Nz, stl_solids=stl_solids, stl_materials=stl_materials, stl_scale=1.0,
                     stl_method='implicit_distance_tol')

testgrid = gfit.grid
surf = gfit.read_stl('Fingers')
spacing = [gfit.x[1] - gfit.x[0], gfit.y[1] - gfit.y[0], gfit.z[1] - gfit.z[0]]

vol, area, errvol, errarea = benchmark(testgrid, spacing, surf_finger)
print(f"Volume: {vol}, Area: {area}, Volume Error: {errvol}, Area Error: {errarea}")




In [ ]:

# threshold only works on a mesh object, and the point_data_to_cell_data returns a flat numpy array we must store the cell mask as a mesh/grid (3D)
inside = testgrid.threshold(0.5, scalars='Fingers')

pl = pv.Plotter()
pl.add_mesh(surf, style='wireframe', opacity=0.2)
pl.add_mesh(inside, color='red', show_edges=True)
pl.show()

In [ ]:
# If you just want a thin shell at the boundary:
#inside = testgrid.contour(isosurfaces=[0.1], scalars='Fingers')  
# #countour needs to be on point data, this switch is very inprecise/ineffecient as it is already done reverse in Wakis
grid_with_points = testgrid.cell_data_to_point_data()
inside = grid_with_points.contour(isosurfaces=[0.1], scalars='Fingers')


pl = pv.Plotter()
pl.add_mesh(surf, style='wireframe', opacity=0.2)
pl.add_mesh(inside, color='red', line_width=2)
pl.show()

In [ ]:

pl = pv.Plotter()
pl.add_mesh(surf, style='wireframe', opacity=0.3, color='blue')
slice_mid = testgrid.slice(normal='z')
pl.add_mesh(slice_mid, scalars='Fingers', cmap='viridis', show_edges=True)
pl.show()

## Studying implicit_distance and voxelize_rectilinear for the Fingers

In [6]:
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
surf_finger= pv.read(stl_finger)
xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
Lx, Ly, Lz = (xmax-xmin), (ymax-ymin), (zmax-zmin)
stl_tol=1e-3
Nx, Ny, Nz = 100, 100, 200
x = np.linspace(xmin, xmax, Nx)
y = np.linspace(ymin, ymax, Ny)
z = np.linspace(zmin, zmax, Nz)

stl_solids = { 'Fingers': stl_finger}
stl_materials = {'Fingers': [1e4, 1., 1e4]}

In [7]:

def read_stl( key):
    # import stl
    surf = pv.read(stl_solids[key])
    '''# rotate
    surf = surf.rotate_x(stl_rotate[key][0])
    surf = surf.rotate_y(stl_rotate[key][1])
    surf = surf.rotate_z(stl_rotate[key][2])
    # translate
    surf = surf.translate(stl_translate[key])
    # scale
    surf = surf.scale(stl_scale[key])'''
    return surf


def plot_stl_mask(stl_solid, cmap='viridis', bounding_box=True, show_grid=True,
                      add_stl='all', stl_opacity=0., stl_colors=None,
                      xmax=None, ymax=None, zmax=None,
                      anti_aliasing='ssaa', smooth_shading=False, off_screen=False):

        pv.global_theme.allow_empty_mesh = True
        pl = pv.Plotter()
        vals = {'x':xmax, 'y':ymax, 'z':zmax}

        # --- Update function ---
        def update_clip(val, axis="x"):
            vals[axis] = val
            # define bounds dynamically
            if axis == "x":
                slice_obj = grid.slice(normal="x", origin=(val, 0, 0))
            elif axis == "y":
                slice_obj = grid.slice(normal="y", origin=(0, val, 0))
            else:  # z
                slice_obj = grid.slice(normal="z", origin=(0, 0, val))

            # add clipped volume (scalars)
            pl.add_mesh(
                grid.clip_box(bounds=(xmin, vals['x'],
                                           ymin, vals['y'],
                                           zmin, vals['z']), invert=False),
                scalars=stl_solid,
                cmap=cmap,
                name="clip",
            )

            # add slice wireframe (grid structure)
            if show_grid:
                pl.add_mesh(slice_obj, style="wireframe", color="grey", name="slice")

        # Plot stl surface(s)
        if add_stl is not None:
            if type(add_stl) is str: #add all stl solids
                if add_stl.lower() == 'all':
                    for i, key in enumerate(stl_solids):
                        surf = read_stl(key)
                        if type(stl_colors) is dict:
                            pl.add_mesh(surf, color=stl_colors[key], opacity=stl_opacity, silhouette=dict(color=stl_colors[key]), name=key)
                        elif type(stl_colors) is list:
                            pl.add_mesh(surf, color=stl_colors[i], opacity=stl_opacity, silhouette=dict(color=stl_colors[i]), name=key)
                        else:
                            pl.add_mesh(surf, color='white', opacity=stl_opacity, silhouette=True, name=key)
                else: #add 1 selected stl solid
                    key = add_stl
                    surf = read_stl(key)
                    pl.add_mesh(surf, color=stl_colors[key], opacity=stl_opacity, silhouette=dict(color=stl_colors[key]), name=key)

            elif type(add_stl) is list: #add selected list of stl solids
                for i, key in enumerate(add_stl):
                    surf = read_stl(key)
                    if type(stl_colors[key]) is dict:
                        pl.add_mesh(surf, color=stl_colors[key], opacity=stl_opacity, silhouette=dict(color=stl_colors[key]), name=key)
                    elif type(stl_colors) is list:
                        pl.add_mesh(surf, color=stl_colors[i], opacity=stl_opacity, silhouette=dict(color=stl_colors[i]), name=key)
                    else:
                        pl.add_mesh(surf, color='white', opacity=stl_opacity, silhouette=True, name=key)

        # --- Sliders (placed side-by-side vertically) ---
        pl.add_slider_widget(
            lambda val: update_clip(val, "x"),
            [xmin, xmax],
            value=xmax, title="X Clip",
            pointa=(0.8, 0.8), pointb=(0.95, 0.8),  # top-right
            style='modern',
        )

        pl.add_slider_widget(
            lambda val: update_clip(val, "y"),
            [ymin, ymax],
            value=ymax, title="Y Clip",
            pointa=(0.8, 0.6), pointb=(0.95, 0.6),  # middle-right
            style='modern',
        )

        pl.add_slider_widget(
            lambda val: update_clip(val, "z"),
            [zmin, zmax],
            value=zmax, title="Z Clip",
            pointa=(0.8, 0.4), pointb=(0.95, 0.4),  # lower-right
            style='modern',
        )

        # Camera orientation
        pl.camera_position = 'zx'
        pl.camera.azimuth += 30
        pl.camera.elevation += 30
        pl.set_background('mistyrose', top='white')
        #self._add_logo_widget(pl)
        pl.add_axes()
        pl.enable_3_lights()
        pl.enable_anti_aliasing(anti_aliasing)

        if bounding_box:
            pl.add_mesh(pv.Box(bounds=(xmin, xmax, ymin, ymax, zmin, zmax)),
            style="wireframe", color="black", line_width=2, name="domain_box")

        if off_screen:
            pl.export_html(f'grid_stl_mask_{stl_solid}.html')
        else:
            pl.show()

In [ ]:
grid = pv.RectilinearGrid(x, y, z)

grid.compute_implicit_distance(surf_finger, inplace=True)
dist = grid["implicit_distance"] 

stl_tol=1e-3
stl_tolerance = np.min([np.min(np.diff(x)), np.min(np.diff(y)), np.min(np.diff(z))]) *stl_tol

grid["Fingers"] = (grid["implicit_distance"] <= stl_tolerance).astype(float)
inside_voxels = grid.threshold(0.5, scalars="Fingers")

grid_cells = grid.point_data_to_cell_data()
#benchmark(grid_cells, spacing, surf_shell)



In [12]:

pl = pv.Plotter()
plane = pv.Plane()
plane.compute_implicit_distance(surf_finger, inplace=True)

pl.add_mesh(plane,scalars='implicit_distance', cmap='bwr', clim=[-0.5, 0.5])

pl.add_mesh(surf_finger, color='w', style='wireframe', opacity=0.5)

pl.show()

Widget(value='<iframe src="http://localhost:33365/index.html?ui=P_0x7f9b5d8d9690_4&reconnect=auto" class="pyvi…

In [ ]:
stl_finger= 'clara_stl_files/012_LHC_WVM_6L2-cube.stl'
stl_solids = { 'Fingers': stl_finger}

plot_stl_mask(stl_solid='Fingers', cmap='viridis', bounding_box=True,
              xmax=xmax, ymax=ymax, zmax=zmax,
              show_grid=True, stl_opacity=0.2)

In [ ]:

pl = pv.Plotter()
pl.add_mesh(surf, color='cyan', opacity=0.1, style='wireframe', label='Original STL')
pl.add_mesh(inside_voxels, color='red', show_edges=True, label='Grid Interior')
#pl.add_mesh(dist, color='red', show_edges=True, label='Grid implicit distances')
pl.add_legend()
pl.show()

In [ ]:
grid = pv.RectilinearGrid(x, y, z)
grid.compute_implicit_distance(surf_shell, inplace=True)
grid_cells = grid.point_data_to_cell_data()
key="mask"
grid_cells[key] = (grid_cells["implicit_distance"] <= stl_tolerance).astype(float)

benchmark(grid_cells, spacing, surf_shell)
cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
grid_cells.plot(scalars='mask', show_edges=False, cpos=cpos)



voxelized

In [ ]:
vox = surf.voxelize_rectilinear(spacing=spacing)
testgrid['shell'] = vox.cell_data['mask'].astype(bool)

In [ ]:
xmin, xmax, ymin, ymax, zmin, zmax = surf_finger.bounds
spacing = [(xmax - xmin) / 100, (ymax - ymin) / 100, (zmax - zmin) / 200]   #not exactly the same as the spacing used to create the grid, but should be close enough for testing

voxelized_grid = surf_shell.voxelize_rectilinear(spacing=spacing)

print(voxelized_grid)
print(voxelized_grid['mask'])
print(voxelized_grid.cell_data['mask'])

cpos = pv.CameraPosition(position=(15, 3, 15), focal_point=(0, 0, 0), viewup=(0, 0, 0))
voxelized_grid.plot(scalars='mask', show_edges=False, cpos=cpos)

array_name = 'mask'
is_inside = voxelized_grid.cell_data[array_name].view(np.bool_)

pl = pv.Plotter()
pl.add_mesh(surf_finger, color='cyan', style='wireframe', opacity=0.3,label="Origional STL")
inside_voxels = voxelized_grid.extract_cells(voxelized_grid.cell_data[array_name].view(np.bool_))
pl.add_mesh(inside_voxels, color='red', label="Voxelized Interior")
pl.add_legend()
pl.show()

In [ ]:
def plot_stl_mask(self, stl_solid, cmap='viridis', mode='interactive', 
                      thresholds=[0.1, 0.8], bounding_box=True, show_grid=True,
                      add_stl='all', stl_opacity=0.1, stl_colors=None,
                      xmax=None, ymax=None, zmax=None,
                      anti_aliasing='ssaa', off_screen=False):
        """
        Extended visualization providing both interactive clipping and 
        side-by-side threshold comparisons.
        """

        if mode == 'comparison':
            # Create 1 row, 3 columns
            pl = pv.Plotter(shape=(1, 3), window_size=[1500, 500])
            
            # Ensure we have point data for smooth distance visualization
            grid_p = grid.cell_data_to_point_data()

            # --- Subplot 1: Raw Implicit Distance ---
            pl.subplot(0, 0)
            pl.add_text("Implicit Distance Field", font_size=10)
            pl.add_mesh(grid_p, scalars="implicit_distance", cmap=cmap)
            pl.add_mesh(self.read_stl(stl_solid), color='white', opacity=0.2, style='wireframe')

            # --- Subplot 2: Threshold A (Liberal) ---
            pl.subplot(0, 1)
            pl.add_text(f"Mask (Threshold: {thresholds[0]})", font_size=10)
            mask_a = self.grid.threshold(thresholds[0], scalars=stl_solid)
            pl.add_mesh(mask_a, color='orange', show_edges=show_grid)

            # --- Subplot 3: Threshold B (Strict) ---
            pl.subplot(0, 2)
            pl.add_text(f"Mask (Threshold: {thresholds[1]})", font_size=10)
            mask_b = self.grid.threshold(thresholds[1], scalars=stl_solid)
            pl.add_mesh(mask_b, color='red', show_edges=show_grid)

            # Link all views so they rotate together
            pl.link_views()
            
        else:
            # --- Original Interactive Logic ---
            pl = pv.Plotter()
            vals = {'x': xmax, 'y': ymax, 'z': zmax}

            def update_clip(val, axis="x"):
                vals[axis] = val
                # Re-calculate clip based on current slider values
                clipped = self.grid.clip_box(bounds=(self.xmin, vals['x'],
                                                   self.ymin, vals['y'],
                                                   self.zmin, vals['z']), invert=False)
                pl.add_mesh(clipped, scalars=stl_solid, cmap=cmap, name="clip")
                
                if show_grid:
                    # Dynamic slice at the slider position
                    origin = [val if axis == a else 0 for a in "xyz"]
                    slice_obj = self.grid.slice(normal=axis, origin=origin)
                    pl.add_mesh(slice_obj, style="wireframe", color="grey", name="slice")

            # [Add Sliders and STL logic from your original code here]
            # ...
            
        # Final formatting common to both modes
        pl.add_axes()
        pl.enable_anti_aliasing(anti_aliasing)
        if off_screen:
            pl.export_html(f'grid_comparison_{stl_solid}.html')
        else:
            pl.show()